# SGLang + ASH-KV RadixAttention Hardware Simulator
This notebook tests the SGLang integration using **real Triton kernels** and **real CUDA memory allocations** on a Google Colab T4 GPU.

We will simulate a 4000-token physical memory limit and watch ASH-KV seamlessly intercept evictions to compress them into INT8 shadow memory.


In [ ]:
!pip install -q torch triton matplotlib numpy
# Ensure we clone the ASH-KV repo to use its kernels and codecs
!git clone https://github.com/Ijas14/ASH-KV.git
import sys
sys.path.append('/content/ASH-KV')


In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
import torch

from ashkv.contracts import PageTable
from ashkv.sglang_integration.allocator import SGLangShadowAllocator
from ashkv.sglang_integration.hooks import SGLangHooks

# Ensure we are on a GPU
assert torch.cuda.is_available(), "Please enable a T4 GPU in Colab!"


In [ ]:
# =========================================================================
# SGLANG RADIX ATTENTION MOCKS (But using REAL CUDA tensors!)
# =========================================================================

class MockTokenToKVPool:
    def __init__(self, capacity_tokens: int):
        self.capacity_tokens = capacity_tokens
        self.allocated_tokens = 0

    def allocate(self, num_tokens: int):
        if self.allocated_tokens + num_tokens > self.capacity_tokens:
            return None # OOM
        self.allocated_tokens += num_tokens
        # Return dummy indices for physical tracking
        return torch.arange(num_tokens)

    def free(self, indices: torch.Tensor):
        if indices is not None:
            self.allocated_tokens -= len(indices)

class MockRadixNode:
    def __init__(self, name: str, length: int):
        self.name = name
        self.length = length
        self.kv_indices = None
        self.last_access_time = time.time()
        self.is_compressed = False

class MockRadixCache:
    def __init__(self, pool: MockTokenToKVPool, ashkv_hooks):
        self.pool = pool
        self.hooks = ashkv_hooks
        self.nodes = []
        
    def add_node(self, name: str, length: int) -> MockRadixNode:
        node = MockRadixNode(name, length)
        indices = self.pool.allocate(length)
        
        while indices is None:
            lru_node = None
            for n in self.nodes:
                if not n.is_compressed and n.kv_indices is not None:
                    if lru_node is None or n.last_access_time < lru_node.last_access_time:
                        lru_node = n
                        
            if lru_node is None:
                raise RuntimeError("Absolute OOM! Cannot evict any more nodes.")
                
            print(f"  [SGLang] Pool Full. Evicting LRU Node: {lru_node.name}")
            
            # Using real CUDA tensors for the SGLang KV Cache!
            sglang_cache = torch.rand((32, lru_node.length, 128), dtype=torch.bfloat16, device="cuda")
            
            # This will trigger real Triton kernels!
            success = self.hooks.demote_hook(lru_node, sglang_kv_cache=sglang_cache, memory_pool=self.pool)
            if not success:
                print(f"  [SGLang] Demotion failed, deleting node {lru_node.name}")
                self.pool.free(lru_node.kv_indices)
                lru_node.kv_indices = None
                
            indices = self.pool.allocate(length)
            
        node.kv_indices = indices
        self.nodes.append(node)
        return node
        
    def touch_node(self, node: MockRadixNode):
        node.last_access_time = time.time()
        if node.is_compressed:
            print(f"  [SGLang] Cache Hit on compressed Node {node.name}! Promoting to BF16...")
            sglang_cache = torch.zeros((32, node.length, 128), dtype=torch.bfloat16, device="cuda")
            self.hooks.promote_hook(node, sglang_kv_cache=sglang_cache, memory_pool=self.pool)


In [ ]:
# =========================================================================
# WORKLOAD SIMULATION & VISUALIZATION
# =========================================================================

pt = PageTable()
# Shadow allocator: 100 MB budget on GPU
shadow_alloc = SGLangShadowAllocator(max_bytes=100 * 1024 * 1024)
class DummyConfig:
    num_hidden_layers = 32
hooks = SGLangHooks(pt, shadow_alloc, DummyConfig())

pool = MockTokenToKVPool(capacity_tokens=2000)
cache = MockRadixCache(pool, hooks)

history_bf16 = []
history_int8 = []

print("--- STARTING SGLANG WORKLOAD SIMULATION ---")

print("\n[Tick 1] Generating System Prompt (1000 tokens)")
sys_node = cache.add_node("System Prompt", 1000)
history_bf16.append(pool.allocated_tokens)
history_int8.append(shadow_alloc.allocated_bytes)

user_nodes = []
for i in range(5):
    print(f"\n[Tick 2.{i}] User {i} connecting and generating (400 tokens)")
    u_node = cache.add_node(f"User {i} Chat", 400)
    user_nodes.append(u_node)
    
    cache.touch_node(sys_node)
    
    history_bf16.append(pool.allocated_tokens)
    history_int8.append(shadow_alloc.allocated_bytes)
    time.sleep(0.01)
    
print("\n[Tick 3] User 0 returns to their chat (Prefix Hit)")
cache.touch_node(user_nodes[0])
history_bf16.append(pool.allocated_tokens)
history_int8.append(shadow_alloc.allocated_bytes)

# Visualization
int8_token_equivalent = [b / 4096 for b in history_int8]

plt.figure(figsize=(10, 6))
plt.fill_between(range(len(history_bf16)), history_bf16, label="SGLang Physical VRAM (BF16)", alpha=0.7, color='blue')
plt.fill_between(range(len(int8_token_equivalent)), [b + i for b, i in zip(history_bf16, int8_token_equivalent)], history_bf16, label="ASH-KV Shadow VRAM (INT8)", alpha=0.7, color='orange')
plt.axhline(y=2000, color='r', linestyle='--', label="Physical VRAM Limit (2000 tokens)")

plt.title("SGLang + ASH-KV Hardware Simulation (Colab GPU)")
plt.xlabel("Simulation Ticks (Requests)")
plt.ylabel("Stored Tokens (Equivalent)")
plt.legend(loc="upper left")
plt.grid(True, alpha=0.3)
plt.show()
